In [1]:
import pickle
%pip install litstudy
import litstudy

%pip install torch
import torch

%pip install --upgrade transformers==4.51.3
%pip install -U adapters

import numpy as np
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [21]:
docs_ai = []
docs_nai = []
with open('HEA_Docs_ai.pkl','rb') as f:
#with open('RAC_Docs_ai.pkl','rb') as f:
    docs_ai = pickle.load(f)

with open('HEA_Docs_nai.pkl','rb') as f:
#with open('RAC_Docs_nai.pkl','rb') as f:
    docs_nai = pickle.load(f)

print(len(docs_ai))
print(len(docs_nai))

866
16758


In [22]:
import random
# make random samples 
ai_index = list(range(0,len(docs_ai)) )
nai_index = list(range(0,len(docs_nai)) )
random.shuffle(ai_index) # shuffled 
random.shuffle(nai_index)

ai_n = int(len(docs_ai)/10)
nai_n = int(len(docs_nai)/10)

ai_index_chunks = [ai_index[i:i+ai_n] for i in range(0,len(docs_ai),ai_n)]
nai_index_chunks = [nai_index[i:i+ai_n] for i in range(0,len(docs_ai),ai_n)]

ai_index_chunks = ai_index_chunks[:-1]
nai_index_chunks = nai_index_chunks[:-1]

# split docs into 10 samples each
docs_ai_samples = [docs_ai.select(t) for t in ai_index_chunks]
docs_nai_samples = [docs_nai.select(t) for t in nai_index_chunks]

print(docs_ai_samples)
print(docs_nai_samples)

print(docs_ai_samples[0][0].title)
print(docs_nai_samples[0][0].title)



[<86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>]
[<86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>, <86 documents>]
Microelasticity model of random alloys. Part II: displacement and stress correlations
Fabrication and characterization of Al x CoFeNiCu 1-x high entropy alloys by laser metal deposition


In [32]:
def extract_embedding(docs_ai,docs_nai):
    embedding_list_ai = []
    embedding_list_nai = []

    # load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')

    #load base model
    model = AutoAdapterModel.from_pretrained('allenai/specter2_base')

    #load the adapter(s) as per the required task, provide an identifier for the adapter in load_as argument and activate it
    model.load_adapter("allenai/specter2", source="hf", load_as="specter2", set_active=True)

    # preprocess the input and get the embeddings of AI-papers to a list
    for item in docs_ai:
        row = item.title + item.abstract
        token = tokenizer(row, padding=True, truncation=True,
                                   return_tensors="pt", return_token_type_ids=False, max_length=288)
        input_ids = token["input_ids"].detach().clone()
        embeddings = model.get_input_embeddings()(input_ids) 
    
        # get the shape of each embedding
        [a,b,c] = embeddings.shape
    
        i=0
        while i<b:
            embedding_list_ai.append(embeddings[0][i].detach().numpy())
            i=i+1
    
    print(len(embedding_list_ai[0]),"embedding dimensions of AI papers")

    # preprocess the input and get the embeddings of Non-AI-papers to a list
    for item in docs_nai:
        row = item.title + item.abstract
        token = tokenizer(row, padding=True, truncation=True,
                                   return_tensors="pt", return_token_type_ids=False, max_length=288)
        input_ids = token["input_ids"].detach().clone()
        embeddings = model.get_input_embeddings()(input_ids) 
    
        # get the shape of each embedding
        [a,b,c] = embeddings.shape
    
        i=0
        while i<b:
            embedding_list_nai.append(embeddings[0][i].detach().numpy())
            i=i+1
    
    print(len(embedding_list_nai[0]),"embedding dimensions of non-AI papers")

    return embedding_list_ai, embedding_list_nai

#[embedding_ai, embedding_nai] = extract_embedding(docs_ai, docs_nai)
i = 9
[ai_e, nai_e] = extract_embedding(docs_ai_samples[i],docs_nai_samples[i])
    
n = str(i)
with open('HEA_embed_ai_sample_'+ n +'.pkl','wb') as file:
#with open('RAC_embed_ai_sample_'+ n +'.pkl','wb') as file:
    pickle.dump(ai_e,file)

with open('HEA_embed_nai_sample_'+ n +'.pkl','wb') as file:
#with open('RAC_embed_nai_sample_'+ n +'.pkl','wb') as file:
    pickle.dump(nai_e,file)



Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


768 embedding dimensions of AI papers
768 embedding dimensions of non-AI papers
